In [1]:
# Cell 1: Imports and helper functions

import os
import json
import pandas as pd
import numpy as np

# Percentile helper: ignores NaNs and returns None if series is empty
def pct(series, q):
    s = pd.to_numeric(series, errors="coerce").dropna()
    if s.empty:
        return None
    return float(np.percentile(s, q, method="linear"))

# Safe JSON parse helper
def safe_json_parse(s):
    try:
        return json.loads(s)
    except:
        return {}

print("✓ Imports loaded successfully")

✓ Imports loaded successfully


In [2]:
# Cell 2: CONFIG — PASTE FULL PATHS ONLY

# ============================================================
# EDIT THESE THREE LINES - Paste complete paths
# ============================================================

# Database name (for display in output CSV)
db_name = "AstraDB"

# Full path to the INPUT CSV file (throughput.csv)
mixed_csv_path = r"C:\Users\avyaa\logs\throughput.csv"

# Full path to the OUTPUT FOLDER (where you want mixed_metrics.csv saved)
# You must create this folder manually before running
mixed_out_dir = r"C:\Users\avyaa\Astra_results\Mixed\Data"

# ============================================================
# DO NOT EDIT BELOW
# ============================================================

# Output file paths
mixed_metrics_out = os.path.join(mixed_out_dir, "mixed_metrics.csv")
mixed_validation_out = os.path.join(mixed_out_dir, "mixed_validation.csv")
mixed_per_op_out = os.path.join(mixed_out_dir, "mixed_per_operation.csv")

# Mixed phase expectations
EXPECTED_READ_PCT = 65.0
EXPECTED_WRITE_PCT = 35.0
TOLERANCE_PCT = 5.0  # Allow ±5% tolerance

# Verification
print("Database:", db_name)
print("Input CSV:", mixed_csv_path)
print("Output metrics:", mixed_metrics_out)
print("Output validation:", mixed_validation_out)
print("Output per-op:", mixed_per_op_out)

if not os.path.exists(mixed_csv_path):
    print("\n⚠️ INPUT FILE NOT FOUND! Check your path.")
else:
    print("\n✓ Input file found!")
    
if not os.path.exists(mixed_out_dir):
    print("⚠️ OUTPUT FOLDER NOT FOUND! Create it manually first.")
else:
    print("✓ Output folder exists!")

Database: AstraDB
Input CSV: C:\Users\avyaa\logs\throughput.csv
Output metrics: C:\Users\avyaa\Astra_results\Mixed\Data\mixed_metrics.csv
Output validation: C:\Users\avyaa\Astra_results\Mixed\Data\mixed_validation.csv
Output per-op: C:\Users\avyaa\Astra_results\Mixed\Data\mixed_per_operation.csv

✓ Input file found!
✓ Output folder exists!


In [3]:
# Cell 3: Load throughput CSV and separate summary row

# Required columns per logging strategy
required_cols = [
    "timestamp_utc","run_id","db_system","server_version","driver",
    "connection_params","db_name","sequence_number","operation_id","operation_type",
    "execution_ms","row_count_or_rows_affected","success",
    "error_code","error_message","params_json"
]

df = pd.read_csv(mixed_csv_path)

# Check for missing columns
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Separate SUMMARY row from regular operation rows
df_summary = df[df["operation_id"].astype(str).str.upper() == "SUMMARY"].copy()
df_ops = df[df["operation_id"].astype(str).str.upper() != "SUMMARY"].copy()

print("Total rows:", len(df))
print("Summary rows:", len(df_summary))
print("Operation rows:", len(df_ops))

# Extract summary info if present
if len(df_summary) > 0:
    summary_row = df_summary.iloc[0]
    summary_elapsed = summary_row["execution_ms"]
    summary_params = safe_json_parse(summary_row["params_json"]) if pd.notna(summary_row["params_json"]) else {}
    
    print("\n📊 Summary row found:")
    print(f"  Total elapsed: {summary_elapsed} ms")
    print(f"  Summary params: {summary_params}")
else:
    print("\n⚠️ No SUMMARY row found in throughput.csv")
    summary_elapsed = None
    summary_params = {}

Total rows: 501
Summary rows: 1
Operation rows: 500

📊 Summary row found:
  Total elapsed: 157667.014 ms
  Summary params: {'reads': 325, 'writes': 175, 'QPS': 2.061, 'TPS': 1.11}


In [4]:
# Cell 4: Clean operation rows and validate success flags

# Normalize columns
df_ops["sequence_number"] = pd.to_numeric(df_ops["sequence_number"], errors="coerce")
df_ops["execution_ms"] = pd.to_numeric(df_ops["execution_ms"], errors="coerce")
df_ops["row_count_or_rows_affected"] = pd.to_numeric(df_ops["row_count_or_rows_affected"], errors="coerce")

# Normalize success flag (handle various formats)
df_ops["success"] = df_ops["success"].astype(str).str.lower().isin(["1","true","yes","t"])

# Normalize operation_type
df_ops["operation_type"] = df_ops["operation_type"].astype(str).str.lower()

# Separate success/error rows
df_success = df_ops[df_ops["success"]].copy()
df_error = df_ops[~df_ops["success"]].copy()

print("Operation rows (usable):", len(df_ops))
print("Success operations:", len(df_success))
print("Error operations:", len(df_error))

# Count reads vs writes
read_cnt = int((df_ops["operation_type"] == "read").sum())
write_cnt = int((df_ops["operation_type"] == "write").sum())
total_cnt = len(df_ops)

read_pct = (read_cnt / total_cnt * 100.0) if total_cnt > 0 else 0.0
write_pct = (write_cnt / total_cnt * 100.0) if total_cnt > 0 else 0.0

print(f"\n📊 Mix verification:")
print(f"  Reads: {read_cnt} ({read_pct:.1f}%)")
print(f"  Writes: {write_cnt} ({write_pct:.1f}%)")
print(f"  Total: {total_cnt}")

Operation rows (usable): 500
Success operations: 500
Error operations: 0

📊 Mix verification:
  Reads: 325 (65.0%)
  Writes: 175 (35.0%)
  Total: 500


In [5]:
# Cell 5: Database-level throughput (QPS, TPS, total elapsed)

# Calculate actual elapsed time from operation sequence
if len(df_ops) > 0:
    # Use sequence timestamps if available, else use summary
    first_ts = pd.to_datetime(df_ops["timestamp_utc"].iloc[0], errors="coerce")
    last_ts = pd.to_datetime(df_ops["timestamp_utc"].iloc[-1], errors="coerce")
    
    if pd.notna(first_ts) and pd.notna(last_ts):
        elapsed_seconds = (last_ts - first_ts).total_seconds()
    else:
        # Fallback: use summary elapsed if timestamps are bad
        elapsed_seconds = (summary_elapsed / 1000.0) if summary_elapsed else None
else:
    elapsed_seconds = None

# Compute QPS and TPS
if elapsed_seconds and elapsed_seconds > 0:
    qps = read_cnt / elapsed_seconds
    tps = write_cnt / elapsed_seconds
    total_throughput = (read_cnt + write_cnt) / elapsed_seconds
else:
    qps = None
    tps = None
    total_throughput = None

# Create database-level summary
db_summary = {
    "db_system": db_name,
    "total_operations": total_cnt,
    "read_count": read_cnt,
    "write_count": write_cnt,
    "read_pct": read_pct,
    "write_pct": write_pct,
    "success_count": len(df_success),
    "error_count": len(df_error),
    "elapsed_seconds": elapsed_seconds,
    "read_qps": qps,
    "write_tps": tps,
    "total_throughput_ops_sec": total_throughput,
    "summary_elapsed_ms": summary_elapsed
}

print("\n✓ Database-level metrics computed:")
print(f"  Elapsed: {elapsed_seconds:.2f}s" if elapsed_seconds else "  Elapsed: N/A")
print(f"  Read QPS: {qps:.2f}" if qps else "  Read QPS: N/A")
print(f"  Write TPS: {tps:.2f}" if tps else "  Write TPS: N/A")
print(f"  Total throughput: {total_throughput:.2f} ops/sec" if total_throughput else "  Total throughput: N/A")


✓ Database-level metrics computed:
  Elapsed: 157.29s
  Read QPS: 2.07
  Write TPS: 1.11
  Total throughput: 3.18 ops/sec


In [6]:
# Cell 6: Per-operation percentiles (p50, p95, p99, max) - success rows only

per_op_rows = []
for op_id, g in df_success.groupby("operation_id"):
    op_type = g["operation_type"].iloc[0] if len(g) > 0 else "unknown"
    
    p50 = pct(g["execution_ms"], 50)
    p95 = pct(g["execution_ms"], 95)
    p99 = pct(g["execution_ms"], 99)
    mx = float(np.max(g["execution_ms"])) if len(g) > 0 else None
    avg = float(np.mean(g["execution_ms"])) if len(g) > 0 else None
    
    per_op_rows.append({
        "operation_id": op_id,
        "operation_type": op_type,
        "op_count": int(len(g)),
        "avg_ms": avg,
        "p50_ms": p50,
        "p95_ms": p95,
        "p99_ms": p99,
        "max_ms": mx
    })

if len(per_op_rows) > 0:
    per_op_df = pd.DataFrame(per_op_rows).sort_values("operation_id")
else:
    per_op_df = pd.DataFrame(columns=[
        "operation_id","operation_type","op_count","avg_ms","p50_ms","p95_ms","p99_ms","max_ms"
    ])

per_op_df.insert(0, "db_system", db_name)
per_op_df.to_csv(mixed_per_op_out, index=False)

print("✓ Per-operation metrics saved to:", mixed_per_op_out)
if len(per_op_df) > 0:
    display(per_op_df)
else:
    print("  (No successful operations to display)")

✓ Per-operation metrics saved to: C:\Users\avyaa\Astra_results\Mixed\Data\mixed_per_operation.csv


,db_system,operation_id,operation_type,op_count,avg_ms,p50_ms,p95_ms,p99_ms,max_ms
0,AstraDB,R1,read,100,187.467670,187.7350,196.80085,205.25228,209.636
1,AstraDB,R2,read,20,184.735400,183.5750,191.06165,196.19393,197.477
2,AstraDB,R3,read,20,515.466900,200.1200,1750.48610,2947.91322,3247.270
3,AstraDB,R4,read,110,203.759000,198.2740,217.31560,346.48147,499.553
4,AstraDB,R5,read,35,272.438143,189.8120,643.12440,1576.78842,1868.572
5,AstraDB,R6,read,40,234.144750,208.5665,240.16050,741.55814,964.901
6,AstraDB,W1,write,10,187.438300,185.2500,198.46775,201.75995,202.583
7,AstraDB,W2,write,60,379.774917,377.7075,399.28900,403.27634,403.497
8,AstraDB,W3,write,40,189.921300,189.0705,202.82605,208.27818,210.828
9,AstraDB,W4,write,50,950.063280,942.1775,1326.61580,1345.69234,1346.052


In [7]:
# Cell 7: Validation report (mix ratio, counts, summary alignment)

mix_valid = abs(read_pct - EXPECTED_READ_PCT) <= TOLERANCE_PCT and \
            abs(write_pct - EXPECTED_WRITE_PCT) <= TOLERANCE_PCT

validation = {
    "db_system": db_name,
    "total_operations": total_cnt,
    "read_count": read_cnt,
    "write_count": write_cnt,
    "read_pct_actual": read_pct,
    "write_pct_actual": write_pct,
    "read_pct_expected": EXPECTED_READ_PCT,
    "write_pct_expected": EXPECTED_WRITE_PCT,
    "mix_valid": mix_valid,
    "success_count": len(df_success),
    "error_count": len(df_error),
    "summary_row_present": len(df_summary) > 0
}

val_df = pd.DataFrame([validation])
val_df.to_csv(mixed_validation_out, index=False)

print("✓ Validation saved to:", mixed_validation_out)
display(val_df)

if not mix_valid:
    print(f"\n⚠️ WARNING: Mix ratio outside tolerance!")
    print(f"  Expected: {EXPECTED_READ_PCT}% reads, {EXPECTED_WRITE_PCT}% writes")
    print(f"  Actual: {read_pct:.1f}% reads, {write_pct:.1f}% writes")

✓ Validation saved to: C:\Users\avyaa\Astra_results\Mixed\Data\mixed_validation.csv


,db_system,total_operations,read_count,write_count,read_pct_actual,write_pct_actual,read_pct_expected,write_pct_expected,mix_valid,success_count,error_count,summary_row_present
0,AstraDB,500,325,175,65.0,35.0,65.0,35.0,True,500,0,True


In [8]:
# Cell 8: Pooled database-level percentiles (all operations combined)

if len(df_success) > 0:
    pooled_p50 = pct(df_success["execution_ms"], 50)
    pooled_p95 = pct(df_success["execution_ms"], 95)
    pooled_p99 = pct(df_success["execution_ms"], 99)
    pooled_max = float(np.max(df_success["execution_ms"]))
    pooled_avg = float(np.mean(df_success["execution_ms"]))
else:
    pooled_p50 = pooled_p95 = pooled_p99 = pooled_max = pooled_avg = None

db_summary.update({
    "pooled_avg_ms": pooled_avg,
    "pooled_p50_ms": pooled_p50,
    "pooled_p95_ms": pooled_p95,
    "pooled_p99_ms": pooled_p99,
    "pooled_max_ms": pooled_max
})

print("✓ Pooled database-level percentiles computed:")
print(f"  Avg: {pooled_avg:.2f} ms" if pooled_avg else "  Avg: N/A")
print(f"  p50: {pooled_p50:.2f} ms" if pooled_p50 else "  p50: N/A")
print(f"  p99: {pooled_p99:.2f} ms" if pooled_p99 else "  p99: N/A")

✓ Pooled database-level percentiles computed:
  Avg: 313.16 ms
  p50: 197.18 ms
  p99: 1326.85 ms


In [9]:
# Cell 9: Save final mixed metrics

metrics_df = pd.DataFrame([db_summary])

# Reorder columns for clarity
col_order = [
    "db_system","total_operations","read_count","write_count","read_pct","write_pct",
    "success_count","error_count",
    "elapsed_seconds","read_qps","write_tps","total_throughput_ops_sec",
    "pooled_avg_ms","pooled_p50_ms","pooled_p95_ms","pooled_p99_ms","pooled_max_ms",
    "summary_elapsed_ms"
]
metrics_df = metrics_df.loc[:, col_order]

metrics_df.to_csv(mixed_metrics_out, index=False)

print("✓ Metrics saved to:", mixed_metrics_out)
print("\nFinal Mixed metrics:")
display(metrics_df)

✓ Metrics saved to: C:\Users\avyaa\Astra_results\Mixed\Data\mixed_metrics.csv

Final Mixed metrics:


,db_system,total_operations,read_count,write_count,read_pct,write_pct,success_count,error_count,elapsed_seconds,read_qps,write_tps,total_throughput_ops_sec,pooled_avg_ms,pooled_p50_ms,pooled_p95_ms,pooled_p99_ms,pooled_max_ms,summary_elapsed_ms
0,AstraDB,500,325,175,65.0,35.0,500,0,157.294035,2.066194,1.112566,3.17876,313.160388,197.177,971.32575,1326.84559,3247.27,157667.014


In [10]:
# Cell 10: Sanity checks and summaries

print("=" * 60)
print("MIXED METRICS COMPUTATION COMPLETE")
print("=" * 60)

print("\n📁 Files saved:")
print("  -", mixed_metrics_out)
print("  -", mixed_validation_out)
print("  -", mixed_per_op_out)

print("\n⚠️ Mix ratio validation:")
if mix_valid:
    print(f"  ✓ Mix is valid: {read_pct:.1f}% reads, {write_pct:.1f}% writes")
else:
    print(f"  ⚠️ Mix outside tolerance: {read_pct:.1f}% reads, {write_pct:.1f}% writes")

if len(df_error) > 0:
    print(f"\n⚠️ {len(df_error)} operations failed:")
    print("  Top error codes:")
    error_codes = df_error["error_code"].value_counts().head(5)
    display(error_codes)
else:
    print("\n✓ No errors in mixed phase!")

print("\n⚡ Slowest operations (top 5 by p99):")
if len(per_op_df) > 0:
    display(per_op_df.sort_values("p99_ms", ascending=False).head(5))
else:
    print("  (No operations to display)")

print("\n📊 Throughput summary:")
print(f"  Read QPS: {qps:.2f}" if qps else "  Read QPS: N/A")
print(f"  Write TPS: {tps:.2f}" if tps else "  Write TPS: N/A")
print(f"  Overall: {total_throughput:.2f} ops/sec" if total_throughput else "  Overall: N/A")

MIXED METRICS COMPUTATION COMPLETE

📁 Files saved:
  - C:\Users\avyaa\Astra_results\Mixed\Data\mixed_metrics.csv
  - C:\Users\avyaa\Astra_results\Mixed\Data\mixed_validation.csv
  - C:\Users\avyaa\Astra_results\Mixed\Data\mixed_per_operation.csv

⚠️ Mix ratio validation:
  ✓ Mix is valid: 65.0% reads, 35.0% writes

✓ No errors in mixed phase!

⚡ Slowest operations (top 5 by p99):


,db_system,operation_id,operation_type,op_count,avg_ms,p50_ms,p95_ms,p99_ms,max_ms
2,AstraDB,R3,read,20,515.466900,200.1200,1750.4861,2947.91322,3247.270
4,AstraDB,R5,read,35,272.438143,189.8120,643.1244,1576.78842,1868.572
9,AstraDB,W4,write,50,950.063280,942.1775,1326.6158,1345.69234,1346.052
5,AstraDB,R6,read,40,234.144750,208.5665,240.1605,741.55814,964.901
7,AstraDB,W2,write,60,379.774917,377.7075,399.2890,403.27634,403.497



📊 Throughput summary:
  Read QPS: 2.07
  Write TPS: 1.11
  Overall: 3.18 ops/sec
